# Governor — E0029-QWEN: cross-backend allocation experiment

**Notebook outputs are evidence, not truth.** Every headline number here must be
recomputed from raw artifacts by `scripts/verify_colab_run.py` before it counts.

---

## What this tests

Whether the Governor's allocation mechanism survives a change of reasoning
backend. Everything except the backend is held fixed: benchmark family,
allocation formulation, feature boundary, target, grouped cross-validation,
calibration/evaluation discipline, resource accounting.

## What it does NOT test

- It is **not** a replication of the `gpt-oss-120b` preregistration. That run was
  stopped because a token cap was manufacturing failures, and changing backend
  changes the capability regime.
- It does **not** test maximum attainable accuracy. A weaker model is fine; what
  matters is that some problems are solvable and some are not, so there is
  something to allocate between.
- A positive result here would **not** establish that "Governor works". The
  strongest permitted claim is tied to this benchmark, model, resource axis,
  split, estimator, and awaits its own replication.

## Current scientific status

| | |
|---|---|
| observable ceiling | +0.0588 |
| oracle marginal ranking | +0.0505 [+0.0203, +0.0816] |
| learned ranker (E0027) | +0.0055 [−0.0104, +0.0218] |
| learned ranker (E0028) | +0.0070 [−0.0128, +0.0287] |
| real-LLM advantage | **NOT VERIFIED** |

The allocation formulation is sound — an oracle ranker converts it into a gain
with a CI excluding zero. The learned ranker does not. E0028 diagnosed this as
data starvation, and this experiment asks whether the same pattern appears on a
different backend.

## Why this experiment exists

Two runs have already been abandoned for configuration faults that a short check
would have caught:

- `gpt-oss-120b` at a 2500-token cap: 34% of samples truncated, **42% of those
  produced no code at all**, against 0% of uncapped ones. Because harder problems
  reason longer, the artificial failure rate would have risen with difficulty —
  a confound pointed straight at the quantity being measured.
- Qwen3 with default thinking enabled: 3/3 sampled problems consumed the entire
  budget without emitting code, projecting 191 hours.

Every gate below exists because of a specific failure like these.

## Requirements

| | |
|---|---|
| runtime | preflight ≈ 10 min · pilot ≈ 25 min · full run measured before it starts |
| GPU | **L4 recommended** (Runtime → Change runtime type → L4). T4 works; A100 is overkill for a 1.7B model. If `REQUIRE_GPU` is set and none is present the notebook **fails** rather than falling back. |
| storage | ≈ 2 GB (model weights + artifacts) |
| internet | required — GitHub and Hugging Face |
| Drive | **optional**; local first, Drive only as an archive |

## How to reproduce

```
git checkout <commit printed in section 03>
python scripts/verify_colab_run.py --handoff claude_handoff/
```


## 00 — Experiment identity

Frozen here, read by every later section. No cell hard-codes these.

In [ ]:
EXPERIMENT_ID = "E0029-QWEN"
REPO_URL     = "https://github.com/SYT20/Governor.git"
REPO_REF     = ""          # commit or tag; "" means the default branch
REQUIRE_GPU  = True        # L4 expected; fail loudly rather than fall back to CPU
USE_DRIVE    = False       # optional archive; local storage is primary
PILOT_PROBLEMS, PILOT_SAMPLES = 20, 5

print(f"EXPERIMENT_ID = {EXPERIMENT_ID}")
print(f"REPO_REF      = {REPO_REF or '(default branch)'}")
print(f"REQUIRE_GPU   = {REQUIRE_GPU}")

## 01–03 — One-cell bootstrap: environment, dependencies, repository

Paste-able into a brand-new Colab. Assumes no cwd, no imports, no pip state, no Drive, no PYTHONPATH. Ends with `COLAB_BOOTSTRAP = PASS` or a clear error.

In [ ]:
import os, subprocess, sys, pathlib

WORK = pathlib.Path("/content") if pathlib.Path("/content").is_dir() else pathlib.Path.cwd()
REPO = WORK / "Governor"

if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
if REPO_REF:
    subprocess.run(["git", "checkout", "--quiet", REPO_REF], cwd=REPO, check=True)

os.chdir(REPO)
sys.path.insert(0, str(REPO))

rc = subprocess.run([sys.executable, "scripts/colab_bootstrap.py",
                     "--repo", REPO_URL, "--ref", REPO_REF, "--dest", str(REPO)],
                    cwd=REPO)
if rc.returncode != 0:
    raise SystemExit("bootstrap failed — do not continue; see the output above")

## 04–05 — Text-based source loader and import parity

Loads modules from source **text**, independent of notebook state, then loads the same modules by ordinary import and requires the two to agree. A divergence means the notebook is reading a stale cache rather than the repository.

In [ ]:
from scripts.colab_text_loader import (load_package_module, verify_import_parity,
                                       repo_root, file_sha256)

TARGETS = ["governor.harness.ledger", "governor.harness.traps",
           "governor.harness.drivers", "governor.gate.m2_interface",
           "governor.execution.executor", "governor.phase4.statemgr",
           "governor.execfeedback.sandbox", "governor.execfeedback.preflight",
           "governor.execfeedback.publictests"]

bad = []
for t in TARGETS:
    r = verify_import_parity(t, repo_root())
    print(f"  {'ok  ' if r['ok'] else 'FAIL'}  {t:<40} {r['names']:>3} names  sha {r['source_sha256'][:8]}")
    if not r["ok"]:
        bad.append((t, r["problems"]))

assert not bad, f"import parity failed: {bad}"
print("\n  all modules load identically via text loader and normal import")

## 06–10 — Every preflight gate

Environment, loader parity, restart safety in a fresh subprocess, sandbox containment, information boundary, M2 contract, checkpoint resume, frozen split, and the project's own test suite. Ends with `READY FOR FULL RUN` or names the first failing gate.

In [ ]:
import subprocess, sys, json, pathlib

# capture_output so the gate results appear in THIS cell. Inherited stdout from a
# subprocess does not reliably render in a Colab cell, which previously produced
# "a gate failed; see above" with nothing above it.
rc = subprocess.run([sys.executable, "scripts/colab_preflight.py",
                     *(["--require-gpu"] if REQUIRE_GPU else []),
                     "--json", "results/colab_preflight.json"],
                    capture_output=True, text=True)
print(rc.stdout)
if rc.stderr.strip():
    print("--- stderr ---")
    print(rc.stderr[-3000:])

PREFLIGHT_OK = rc.returncode == 0
rep = pathlib.Path("results/colab_preflight.json")
if rep.exists():
    gates = json.loads(rep.read_text())["gates"]
    failed = [g for g in gates if not g["ok"]]
    if failed:
        print("\nFAILING GATES:")
        for g in failed:
            print(f"  - {g['name']}: {g['detail']}")

print(f"\nPREFLIGHT_OK = {PREFLIGHT_OK}")
if not PREFLIGHT_OK:
    raise SystemExit("DO NOT START FULL RUN — see the failing gates above")


## 08 — Backend: load the real model and run real inference

Not an import check. Loads weights, generates twice to prove reuse, and records load time, latency, token counts and peak memory. If the model does not fit it reports `MODEL DOES NOT FIT` and stops — it never silently swaps model.

In [ ]:
import json, time, pathlib
cfg = json.loads(pathlib.Path("configs/colab_model.json").read_text())
print(json.dumps({k: v for k, v in cfg.items() if not k.startswith("_")}, indent=1)[:600])

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
if REQUIRE_GPU and device != "cuda":
    raise SystemExit("GPU required but unavailable — failing rather than falling back")

t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(cfg["model_name"], revision=cfg["revision"])
import inspect
# transformers 4.56 renamed torch_dtype -> dtype. Negotiate rather than pin.
_p = inspect.signature(AutoModelForCausalLM.from_pretrained).parameters
_dk = "dtype" if "dtype" in _p else "torch_dtype"
model = AutoModelForCausalLM.from_pretrained(
    cfg["model_name"], revision=cfg["revision"],
    **{_dk: getattr(torch, cfg["dtype"])}, device_map=cfg["device_map"])
load_s = time.perf_counter() - t0
print(f"  loaded in {load_s:.1f}s on {device}")

MODEL_META = {"model_name": cfg["model_name"], "revision": cfg["revision"],
              "dtype": cfg["dtype"], "device": device, "load_seconds": load_s}
pathlib.Path("results/colab_model_meta.json").write_text(json.dumps(MODEL_META, indent=1))

## 08b — Sequential vs batched: measured on THIS GPU, not assumed

Batching is normally assumed to win on a GPU. Measured on Apple Silicon via MLX it
**lost** — 0.91x, slightly slower than sequential:

```
sequential    93.3s   1875 tokens   20.1 tok/s
batched      102.6s   2306 tokens   22.5 tok/s     0.91x
```

The token counts show why. A batch runs until its **longest** sequence finishes, so
short completions spend steps generating padding — batched did 1.23x the work for
the same result. With this workload's length distribution (median 212, p95 1128)
that waste is severe.

MLX on unified memory is bandwidth-bound at batch size 1, so there is no idle
compute for batching to fill. A CUDA GPU is the opposite, and the mechanism that
makes batching win there is simply absent on Apple Silicon — so the local result
says nothing about Colab. **This cell measures it here instead of extrapolating.**

It also checks whether batched sampling still yields **independent** samples. If a
shared sampler correlates draws, ten samples per problem stop being independent
and the allocation measurement is corrupted — a correctness failure that would
otherwise look like a speedup.


In [ ]:
import subprocess, sys, json, pathlib

subprocess.run([sys.executable, "scripts/colab_batch_bench.py", "--n", "8", "--cap", "1024"])

bench = json.loads(pathlib.Path("results/batch_bench.json").read_text())
GENERATION_MODE = bench["verdict"]          # SEQUENTIAL or BATCHED — measured, not chosen
print(f"\nGENERATION_MODE = {GENERATION_MODE}")
print(f"projected {bench['projected_hours_4750']:.1f} h for 4750 samples on this hardware")
if bench["padding_waste"] > 1.3:
    print(f"note: padding waste {bench['padding_waste']:.2f}x — sort prompts by length before batching")


## 11 — Pilot, and the truncation guard

20 problems × 5 samples. Measures the completion-length distribution and **chooses the generation ceiling from evidence**. This gate exists because a 2500-token cap once produced 42% empty outputs while 0% of uncapped samples did.

In [ ]:
import subprocess, sys, json, pathlib

# Batched, length-bucketed, checkpointed. 20 problems x 5 samples.
rc = subprocess.run([sys.executable, "scripts/e0029_colab_generate.py",
                     "--pilot", str(PILOT_PROBLEMS), str(PILOT_SAMPLES),
                     "--batch-size", "32"])
PILOT_OK = rc.returncode == 0
print(f"\nPILOT_OK = {PILOT_OK}")
if not PILOT_OK:
    raise SystemExit("pilot failed — full run refused; see the gate above")


## 12–13 — Full experiment with continuous checkpointing

Every sample is appended to JSONL immediately and flushed. A restart detects completed samples and skips them; a torn final line is survivable. No sample is ever written twice.

In [ ]:
import subprocess, sys, json, pathlib

rep = json.loads(pathlib.Path("results/E0029-QWEN-preflight.json").read_text())
assert rep.get("ok"), f"pilot did not pass: {rep.get('problems')}"

# Resumable: re-running after a disconnect skips completed samples.
rc = subprocess.run([sys.executable, "scripts/e0029_colab_generate.py",
                     "--full", "--batch-size", "32"])
print(f"\nFULL_RUN_EXIT = {rc.returncode}")
n = sum(1 for _ in open("results/e0029_colab_generations.jsonl"))
print(f"samples on disk: {n} / {475*10}")


## 14–16 — Metrics, statistics, guardrails

Every metric is regenerated from raw JSONL. No headline number is ever carried forward from a printed value.

In [ ]:
# Policies compared through the SAME executor and accounting layer:
#   best fixed · random · myopic · learned Governor · oracle marginal ranking
# Primary:   Governor − best fixed, at matched realised inference tokens
# Secondary: Governor − myopic
# Reported:  paired mean, bootstrap 95% CI, paired sd, AUC, P@5/10, recall@10,
#            lift@10, NDCG, power
print("see scripts/e0028_marginal_ranker.py — the identical, frozen analysis path")

## 17–19 — Reproducibility, report, handoff

Recomputes headline metrics from the saved checkpoint and requires an exact match to the declared tolerance, then writes the handoff package.

## 17–19 — Archive to Drive, and the handoff for verification

**Run this cell as often as you like** — during generation, after it, or after an
interrupt. A partial archive is useful; a missing one is not, and the likeliest
end to a multi-hour Colab session is an unplanned one.

**The mount happens here, in the notebook, and that is not incidental.**
`google.colab.drive` is injected into the notebook *kernel*, and mounting needs
the kernel's interactive channel to show the OAuth prompt. A helper script run as
a subprocess has neither, so a mount attempted from there can only fail — and
would fail reporting "not running in Colab" on a machine that plainly is. The
notebook mounts; `colab_archive.py` only *verifies*, by writing a probe file
rather than trusting that the directory exists.

Drive stays optional. Everything is written locally first, and if the mount is
absent or unwritable the archive still completes and reports
`DRIVE_ARCHIVE = UNAVAILABLE`. Claiming persistence that did not happen is worse
than having none, because you find out only once the runtime is gone.


In [ ]:
import subprocess, sys, json, pathlib

# MOUNT HERE, in-process. See the note above: a subprocess cannot do this.
DRIVE_MOUNTED = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_MOUNTED = pathlib.Path("/content/drive/MyDrive").is_dir()
        print(f"drive mounted: {DRIVE_MOUNTED}")
    except ImportError:
        print("not running in Colab — Drive archive skipped, local archive still runs")
    except Exception as e:
        print(f"mount failed ({type(e).__name__}: {e}) — local archive still runs")

subprocess.run([sys.executable, "scripts/colab_archive.py"]
               + (["--drive"] if (USE_DRIVE and DRIVE_MOUNTED) else []))

hp = pathlib.Path("claude_handoff")
if hp.exists():
    s = json.loads((hp / "experiment_summary.json").read_text())
    d = json.loads((hp / "drive_status.json").read_text())
    print(f"\nSTATUS         {s['status']}")
    print(f"rows           {s['rows']}")
    print(f"problems       {s['problems_seen']}/{s.get('problems_expected')}")
    print(f"DRIVE_ARCHIVE  {d['status']}   {d.get('path', d.get('reason',''))}")
    print("\nVerify locally (the notebook's numbers are evidence, not truth):")
    print("  python scripts/verify_colab_run.py --handoff claude_handoff/")
